# ELISA Statistical Analysis

**Portfolio project: ELISA Analytics & Decision-Support Workflow**

This notebook contains the Python analyses used after SQL-based data integration and rule-based QC.

Workflow covered here:

1. Load the analysis-ready specimen dataset.
2. Restrict performance-factor analysis to QC-valid plates.
3. Explore specimen/process factors statistically.
4. Fit 4-parameter logistic (4PL) calibration curves and assess model fit.
5. Explore specimen replicate CV by plate position.
6. Summarize the evidence without treating descriptive differences as causal findings.

> The dataset is synthetic and intended for portfolio demonstration. Statistical results are interpreted as evidence for investigation, not proof of root cause.


## 1. Setup and data loading

`Final_Specimen_Metadata.csv` is the integrated specimen-level dataset created through the SQL workflow.  
`Calibrators.csv` and `Sample_Wells.csv` are source datasets used for 4PL modelling and exploratory plate-position analysis.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.optimize import curve_fit
from scipy.stats import (
    mannwhitneyu,
    kruskal,
    chi2_contingency,
    fisher_exact,
)

DATA_DIR = Path("../Data")
PROCESSED_DIR = DATA_DIR / "processed"
SOURCE_DIR = DATA_DIR / "source"

specimen_df = pd.read_csv(PROCESSED_DIR / "Final_Specimen_Metadata.csv")
calibrators = pd.read_csv(SOURCE_DIR / "Calibrators.csv")
sample_wells = pd.read_csv(SOURCE_DIR / "Sample_Wells.csv")

print("Specimen dataset:", specimen_df.shape)
print("Calibrators:", calibrators.shape)
print("Sample wells:", sample_wells.shape)


In [ ]:
# Convert date fields used in the analysis.
date_columns = ["CollectionDate", "ReceiptDate", "RunDate"]
specimen_df[date_columns] = specimen_df[date_columns].apply(
    pd.to_datetime,
    format="%d-%m-%Y"
)

# Basic dataset checks.
print("Duplicate specimen/plate records:",
      specimen_df[["SPECIMENID", "PLATEID"]].duplicated().sum())

print("\nFinal specimen status:")
print(specimen_df["SPECIMEN_STATUS"].value_counts())


SPECIMEN_STATUS
PASS                126
PLATE QC FAILED      48
SAMPLE QC FAILED     11
REVIEW SAMPLE QC      7
Name: count, dtype: int64

## 2. Restrict downstream factor analysis to QC-valid plates

Plate validity is established in SQL before specimen/process factors are investigated.  
Specimens from failed plates are therefore excluded from downstream performance-factor analysis.


In [ ]:
valid_df = specimen_df[
    specimen_df["PLATE_QC_STATUS"].str.upper() == "PASS"
].copy()

print("QC-valid specimen records:", valid_df.shape)
print(valid_df["CV_PERCENT"].describe())


## 3. Specimen and process-factor analysis

### 3.1 Matrix

The Mann–Whitney U test compares specimen replicate CV distributions between serum and K2EDTA plasma without assuming normality.


In [ ]:
plasma_cv = valid_df.loc[
    valid_df["Matrix"] == "K2EDTA Plasma", "CV_PERCENT"
]

serum_cv = valid_df.loc[
    valid_df["Matrix"] == "Serum", "CV_PERCENT"
]

print(valid_df.groupby("Matrix")["CV_PERCENT"].describe())

u_stat, p_value = mannwhitneyu(
    plasma_cv,
    serum_cv,
    alternative="two-sided"
)

print("\nMann–Whitney U statistic:", u_stat)
print("p-value:", p_value)


U statistic: 2105.0
p-value: 0.45899890412939726


**Interpretation:** The original notebook result (`p ≈ 0.459`) did not provide evidence of a difference in replicate CV distribution between serum and K2EDTA plasma in this dataset.


### 3.2 Receipt delay

Receipt delay was examined in two ways:

- CV% distribution across 0-, 1- and 2-day receipt-delay groups using Kruskal–Wallis.
- PASS / REVIEW / FAIL outcome distribution using a chi-square test.


In [ ]:
day0_cv = valid_df.loc[valid_df["Receipt_Delay"] == 0, "CV_PERCENT"]
day1_cv = valid_df.loc[valid_df["Receipt_Delay"] == 1, "CV_PERCENT"]
day2_cv = valid_df.loc[valid_df["Receipt_Delay"] == 2, "CV_PERCENT"]

h_stat, p_value = kruskal(day0_cv, day1_cv, day2_cv)

print(valid_df.groupby("Receipt_Delay")["CV_PERCENT"].describe())
print("\nKruskal–Wallis H statistic:", h_stat)
print("p-value:", p_value)


H-statistic: 5.07515854119864
p_value: 0.07905754526576811


In [ ]:
receipt_table = pd.crosstab(
    valid_df["Receipt_Delay"],
    valid_df["SAMPLE_QC_RESULT"]
)

chi2, p_value, dof, expected = chi2_contingency(receipt_table)

print(receipt_table)
print("\nChi-square:", round(chi2, 3))
print("p-value:", round(p_value, 4))
print("Degrees of freedom:", dof)


Chi-square: 6.331
p-value: 0.1758
Degrees of freedom: 4

Expected counts:
[[ 4.65972222 53.375       2.96527778]
 [ 3.4375     39.375       2.1875    ]
 [ 2.90277778 33.25        1.84722222]]


**Interpretation:** Receipt delay showed a descriptive pattern worth monitoring, but neither the Kruskal–Wallis result (`p ≈ 0.079`) nor the QC-outcome chi-square result (`p ≈ 0.176`) met the conventional 0.05 significance threshold.


### 3.3 Reagent kit lot

The original analysis first compared CV distributions by reagent lot, then grouped REVIEW and FAIL outcomes as `QC_CONCERN` for a 2×2 Fisher exact test.


In [ ]:
a17_cv = valid_df.loc[
    valid_df["ReagentKitLot"] == "RK26A17", "CV_PERCENT"
]
b04_cv = valid_df.loc[
    valid_df["ReagentKitLot"] == "RK26B04", "CV_PERCENT"
]

u_stat, p_value = mannwhitneyu(
    a17_cv,
    b04_cv,
    alternative="two-sided"
)

print("Mann–Whitney U statistic:", u_stat)
print("p-value:", p_value)


U statistic: 2584.0
p-value: 0.236213580531484


In [ ]:
valid_df["QC_Group"] = np.where(
    valid_df["SAMPLE_QC_RESULT"] == "PASS",
    "PASS",
    "QC_CONCERN"
)

lot_table = pd.crosstab(
    valid_df["ReagentKitLot"],
    valid_df["QC_Group"]
)

odds_ratio, p_value = fisher_exact(lot_table)

print(lot_table)
print("\nOdds ratio:", round(odds_ratio, 3))
print("p-value:", round(p_value, 4))


Odds ratio: 0.36
p-value: 0.1796


**Interpretation:** The lot-level difference remained descriptive in this small dataset. The Fisher exact test (`p ≈ 0.180`) did not provide sufficient evidence of an association between reagent lot and QC concern.


### 3.4 Operator

Operator was evaluated against the binary `PASS` versus `QC_CONCERN` grouping using a chi-square test.


In [ ]:
operator_table = pd.crosstab(
    valid_df["OperatorID"],
    valid_df["QC_Group"]
)

chi2, p_value, dof, expected = chi2_contingency(operator_table)

print(operator_table)
print("\nChi-square:", round(chi2, 3))
print("p-value:", round(p_value, 4))
print("Degrees of freedom:", dof)


Chi-square: 1.27
p-value: 0.53
Degrees of freedom: 2

Expected counts:
[[63.  9.]
 [42.  6.]
 [21.  3.]]


**Interpretation:** The operator comparison (`p = 0.53`) did not provide evidence of operator-associated QC performance in this dataset.


## 4. 4PL calibration-curve modelling

Mean calibrator OD is calculated at each concentration for each plate. A four-parameter logistic model is then fitted independently to each plate.

Model assessment is kept separate from overall plate validity: a strong calibration fit does not by itself establish that all plate-level QC requirements passed.


In [ ]:
cal_mean = (
    calibrators
    .groupby(["PlateID", "Concentration_U_mL"])["OD450"]
    .mean()
    .reset_index(name="Mean_OD450")
)

def four_pl(x, A, B, C, D):
    return D + (A - D) / (1 + (x / C) ** B)

results = []

for plate, group in cal_mean.groupby("PlateID"):
    x = group["Concentration_U_mL"].to_numpy()
    y = group["Mean_OD450"].to_numpy()

    params, covariance = curve_fit(
        four_pl,
        x,
        y,
        bounds=(0, np.inf),
        maxfev=10000
    )

    predicted_od = four_pl(x, *params)
    residuals = y - predicted_od

    ss_residual = np.sum(residuals ** 2)
    ss_total = np.sum((y - np.mean(y)) ** 2)

    r_squared = 1 - (ss_residual / ss_total)
    rmse = np.sqrt(np.mean(residuals ** 2))

    results.append({
        "PlateID": plate,
        "R_squared": r_squared,
        "RMSE": rmse
    })

curve_fit_results = pd.DataFrame(results)
curve_fit_results


,PlateID,R_squared,RMSE
0,PLT-26-041,0.999890,0.008429
1,PLT-26-042,0.999330,0.020751
2,PLT-26-043,0.999804,0.011300
3,PLT-26-044,0.999551,0.017049
4,PLT-26-045,0.999973,0.004185
5,PLT-26-046,0.999737,0.012956
6,PLT-26-047,0.999917,0.007336
7,PLT-26-048,0.999831,0.010366


**Result:** All eight plates showed very strong 4PL fits (`R² > 0.9993`), with RMSE values of approximately `0.004–0.021`.

These metrics assess calibration-curve fit only. Plate validity also depends on the predefined blank, control and calibrator QC requirements implemented in SQL.


## 5. Exploratory plate-position analysis

Well IDs are mapped to row and column coordinates. Specimen-level replicate CV is then summarized by the starting position of each replicate pair.

This is an exploratory spatial analysis only. The synthetic plate layout was **not designed as a formal edge-effect experiment**, so the analysis should not be interpreted as an edge-effect test.


In [ ]:
sample_wells["Row"] = sample_wells["Well"].str[0]
sample_wells["Column"] = sample_wells["Well"].str[1:].astype(int)

position_qc = sample_wells.merge(
    specimen_df[
        ["PLATEID", "SPECIMENID", "CV_PERCENT", "SAMPLE_QC_RESULT"]
    ],
    left_on=["PlateID", "SpecimenID"],
    right_on=["PLATEID", "SPECIMENID"],
    how="left"
)

specimen_position_qc = (
    position_qc
    .groupby(
        ["PlateID", "SpecimenID", "CV_PERCENT", "SAMPLE_QC_RESULT"],
        as_index=False
    )
    .agg(
        Row=("Row", "first"),
        Start_Column=("Column", "min"),
        End_Column=("Column", "max")
    )
)

valid_position_qc = specimen_position_qc[
    ~specimen_position_qc["PlateID"].isin(
        ["PLT-26-046", "PLT-26-048"]
    )
].copy()

mean_cv_position = (
    valid_position_qc
    .groupby(["Row", "Start_Column"])["CV_PERCENT"]
    .mean()
    .unstack()
)

mean_cv_position


In [ ]:
plt.figure(figsize=(9, 4))
plt.imshow(mean_cv_position, aspect="auto")
plt.colorbar(label="Mean CV (%)")

plt.xticks(
    range(len(mean_cv_position.columns)),
    mean_cv_position.columns
)
plt.yticks(
    range(len(mean_cv_position.index)),
    mean_cv_position.index
)

plt.xlabel("Replicate Pair Start Column")
plt.ylabel("Plate Row")
plt.title("Mean Sample Replicate CV by Plate Position\nValid Plates Only")
plt.show()


**Interpretation:** The exploratory view can identify isolated higher-CV positions, but the current experimental design does not support a formal edge-effect conclusion. A purpose-designed plate-layout study would be required for that assessment.


## 6. Evidence summary

| Question | Statistical approach | Result from project dataset | Interpretation |
|---|---|---:|---|
| Matrix vs replicate CV | Mann–Whitney U | p ≈ 0.459 | No evidence of a CV-distribution difference |
| Receipt delay vs replicate CV | Kruskal–Wallis | p ≈ 0.079 | Descriptive signal; insufficient evidence at 0.05 |
| Receipt delay vs QC outcome | Chi-square | p ≈ 0.176 | No statistically supported association |
| Reagent lot vs replicate CV | Mann–Whitney U | p ≈ 0.236 | No evidence of a CV-distribution difference |
| Reagent lot vs QC concern | Fisher exact | p ≈ 0.180 | Descriptive lot difference only |
| Operator vs QC concern | Chi-square | p = 0.53 | No evidence of operator-associated QC performance |

### Analytical conclusion

The Python analysis supports a disciplined progression from **descriptive signal → statistical assessment → scientific interpretation**. In this small synthetic dataset, selected process/specimen factors generated useful investigation signals, but the statistical tests did not support strong association or causal claims.

The 4PL analysis demonstrated strong calibration-curve fit across plates, while overall plate validity remained governed by the broader QC rules implemented in SQL.
